# 🚗 Smart Parking Intelligence System
## Hackathon Prototype — ML Model for Parking Violation Severity Prediction

**Dataset:** Bangalore Parking Violations (Nov 2023 – Apr 2024)  
**Target:** Predict `severity_level` → Low | Medium | High | Critical  
**Models:** RandomForest · GradientBoosting · ExtraTrees

---

In [ ]:
# ── Cell 1: Imports ─────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')
import os, pickle, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from datetime import datetime

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

SEED = 42; np.random.seed(SEED)
OUTPUT_DIR = 'outputs'; os.makedirs(OUTPUT_DIR, exist_ok=True)
LABEL_ORDER = ['Low', 'Medium', 'High', 'Critical']
FEATURE_COLS = [
    'location_type','illegal_vehicle_count','traffic_volume','average_speed',
    'parking_occupancy','road_width','historical_violation_count','nearby_event',
    'day_of_week','time_of_day','violation_score','vehicle_weight',
    'delay_score','is_weekend','hour','pci_score'
]
print('✅ Imports OK')

## 1. Load Raw Data & Initial EDA

In [ ]:
# ── Cell 2: Load data ───────────────────────────────────────────────────────
DATA_PATH = '/mnt/user-data/uploads/jan_to_may_police_violation_anonymized791b166.csv'

df_raw = pd.read_csv(DATA_PATH)
df_raw['created_datetime'] = pd.to_datetime(df_raw['created_datetime'], format='mixed', utc=True)
# Sample for notebook speed (remove sample() in production for full 298K rows)
df = df_raw.sample(n=80000, random_state=SEED).reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Date range: {df.created_datetime.min()} → {df.created_datetime.max()}')
df.head(3)

In [ ]:
# ── Cell 3: Missing values ───────────────────────────────────────────────────
print('Missing values:')
miss = df.isnull().sum()
print(miss[miss > 0])

In [ ]:
# ── Cell 4: Violation type distribution ─────────────────────────────────────
print('Top violation types:')
print(df['violation_type'].value_counts().head(10).to_string())
print()
print('Vehicle type distribution:')
print(df['vehicle_type'].value_counts().head(8).to_string())

## 2. Feature Engineering

In [ ]:
# ── Cell 5: Time features ────────────────────────────────────────────────────
df['hour']        = df['created_datetime'].dt.hour
df['day_of_week'] = df['created_datetime'].dt.dayofweek
df['month']       = df['created_datetime'].dt.month
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

def time_of_day(h):
    if h < 6:   return 'Night'
    elif h < 12: return 'Morning'
    elif h < 17: return 'Afternoon'
    elif h < 21: return 'Evening'
    return 'Night'

df['time_of_day'] = df['hour'].map(time_of_day)
print('Time features added:', df[['hour','day_of_week','is_weekend','time_of_day']].head(3).to_string())

In [ ]:
# ── Cell 6: Location type & hotspot features ────────────────────────────────
def loc_type(j):
    j = str(j).lower()
    if 'no junction' in j:                              return 'Street'
    if any(k in j for k in ['metro','station','bus']): return 'Transit Hub'
    if any(k in j for k in ['market','plaza','mall']): return 'Commercial Zone'
    if any(k in j for k in ['hospital','school']):     return 'Institutional Zone'
    return 'Junction'

df['location_type'] = df['junction_name'].map(loc_type)

# Historical violation count per ~250m grid cell
df['lat_grid'] = (df['latitude']  / 0.002).round(0)
df['lon_grid'] = (df['longitude'] / 0.002).round(0)
gc = df.groupby(['lat_grid','lon_grid']).size().reset_index(name='historical_violation_count')
df = df.merge(gc, on=['lat_grid','lon_grid'], how='left')

print('Location types:', df['location_type'].value_counts().to_dict())
print('Max historical violations in grid:', df['historical_violation_count'].max())

In [ ]:
# ── Cell 7: Domain-scored features ───────────────────────────────────────────
VIOLATION_SEVERITY = {
    'WRONG PARKING':1,'NO PARKING':2,'PARKING IN A MAIN ROAD':3,
    'PARKING ON FOOTPATH':3,'DOUBLE PARKING':3,
    'PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC':4,
}
VEHICLE_WEIGHT = {
    'SCOOTER':1,'MOPED':1,'MOTOR CYCLE':1,'PASSENGER AUTO':2,
    'GOODS AUTO':2,'CAR':2,'VAN':2,'LGV':3,'MAXI-CAB':3,'PRIVATE BUS':4,
}

def violation_score(v):
    items = re.findall(r'"([^"]+)"', str(v))
    return sum(VIOLATION_SEVERITY.get(x,1) for x in items) if items else 1

df['violation_score'] = df['violation_type'].map(violation_score)
df['vehicle_weight']  = df['vehicle_type'].map(VEHICLE_WEIGHT).fillna(2)
print('Violation score range:', df['violation_score'].min(), '-', df['violation_score'].max())

In [ ]:
# ── Cell 8: Simulated continuous features ────────────────────────────────────
# These are domain-calibrated synthetic features built from real signal
rng = np.random.default_rng(SEED)
n = len(df)

peak = np.where(df['hour'].between(8,10)|df['hour'].between(17,20), 1.4, 1.0)
df['traffic_volume'] = (rng.integers(200,800,n)*peak + df['historical_violation_count'].clip(upper=200)*0.5).astype(int).clip(100,2000)
df['average_speed']  = (60 - df['traffic_volume']/50 + rng.normal(0,3,n)).clip(5,60).round(1)
df['parking_occupancy'] = (40 + df['violation_score']*5 + df['is_weekend']*8 + rng.normal(0,10,n)).clip(0,100).round(1)

road_map = {'Junction':14.0,'Commercial Zone':12.0,'Transit Hub':16.0,'Institutional Zone':12.0,'Street':8.0}
df['road_width'] = (df['location_type'].map(road_map).fillna(10.0) + rng.normal(0,1.5,n)).clip(4,24).round(1)

ep = np.where(df['is_weekend'], 0.3, 0.1)
df['nearby_event'] = rng.binomial(1, ep).astype(int)
df['illegal_vehicle_count'] = (df['vehicle_weight'] + df['violation_score'] + rng.integers(0,5,n) + df['nearby_event']*2).clip(1,20)
df['delay_score'] = ((60-df['average_speed'])/60 * (df['traffic_volume']/2000) * 100).clip(0,100).round(2)

print(df[['traffic_volume','average_speed','parking_occupancy','illegal_vehicle_count','delay_score']].describe().round(2).to_string())

## 3. Parking Congestion Index (PCI)

In [ ]:
# ── Cell 9: PCI computation ──────────────────────────────────────────────────
# PCI = 0.4×norm(illegal_vehicle_count) + 0.3×norm(traffic_volume)
#       + 0.2×norm(delay_score) + 0.1×norm(parking_occupancy)

pci_cols = ['illegal_vehicle_count','traffic_volume','delay_score','parking_occupancy']
pci_scaler = MinMaxScaler()
normed = pci_scaler.fit_transform(df[pci_cols])
df['pci_score'] = (normed * [0.4, 0.3, 0.2, 0.1]).sum(axis=1) * 100

def pci_category(p):
    if p <= 25: return 'Low'
    elif p <= 50: return 'Moderate'
    elif p <= 75: return 'High'
    return 'Critical'

df['pci_category'] = df['pci_score'].map(pci_category)
print('PCI Category distribution:')
print(df['pci_category'].value_counts().reindex(['Low','Moderate','High','Critical']).to_string())

## 4. Target Variable — Severity Level

In [ ]:
# ── Cell 10: Severity label assignment ───────────────────────────────────────
score = (
    df['violation_score']          * 0.30
    + df['vehicle_weight']         * 0.10
    + df['illegal_vehicle_count']  * 0.15
    + df['traffic_volume']  / 200  * 0.15
    + df['parking_occupancy'] / 20 * 0.10
    + (1 - df['average_speed']/60) * 0.10
    + df['nearby_event']           * 0.05
    + np.log1p(df['historical_violation_count']) * 0.05
)
score = (score - score.min()) / (score.max() - score.min()) * 100

df['severity_level'] = pd.cut(score, bins=[-1,25,50,75,101], labels=['Low','Medium','High','Critical'])
print('Severity distribution:')
print(df['severity_level'].value_counts().reindex(LABEL_ORDER).to_string())

## 5. Exploratory Data Analysis Plots

In [ ]:
# ── Cell 11: EDA visualizations ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Smart Parking Intelligence — EDA Dashboard', fontsize=14, fontweight='bold')
pal = ['#2196F3','#FF5722','#4CAF50','#FF9800']

# 1. Severity distribution
ax = axes[0,0]
df['severity_level'].value_counts().reindex(LABEL_ORDER).plot(kind='bar', ax=ax, color=pal, edgecolor='white')
ax.set_title('Severity Level Distribution'); ax.tick_params(axis='x', rotation=0)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x()+p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=8)

# 2. Violations by hour
ax = axes[0,1]
df.groupby('hour').size().plot(ax=ax, color='#2196F3', linewidth=2, marker='o', markersize=3)
ax.set_title('Violations by Hour of Day'); ax.set_xlabel('Hour')

# 3. Day of week
ax = axes[0,2]
days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
df.groupby('day_of_week').size().reindex(range(7)).plot(kind='bar', ax=ax, color='#FF5722')
ax.set_title('Violations by Day of Week'); ax.set_xticklabels(days, rotation=0)

# 4. PCI categories
ax = axes[1,0]
df['pci_category'].value_counts().reindex(['Low','Moderate','High','Critical']).plot(
    kind='bar', ax=ax, color=pal, edgecolor='white')
ax.set_title('PCI Category Distribution'); ax.tick_params(axis='x', rotation=0)

# 5. Location type
ax = axes[1,1]
df['location_type'].value_counts().sort_values().plot(kind='barh', ax=ax, color='#9C27B0')
ax.set_title('Violations by Location Type')

# 6. Traffic volume dist
ax = axes[1,2]
df['traffic_volume'].hist(ax=ax, bins=30, color='#00BCD4', edgecolor='white')
ax.set_title('Traffic Volume Distribution')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/eda_overview.png', dpi=150)
plt.show()

## 6. Preprocessing

In [ ]:
# ── Cell 12: Encode & split ───────────────────────────────────────────────────
cat_cols = ['location_type', 'time_of_day']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

X = df[FEATURE_COLS].fillna(0).values
le_y = LabelEncoder(); le_y.fit(LABEL_ORDER)
y = le_y.transform(df['severity_level'].astype(str))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
print(f'Train: {X_tr.shape}  |  Test: {X_te.shape}')
print('Class balance in test:', {le_y.inverse_transform([i])[0]: int((y_te==i).sum()) for i in range(4)})

## 7. Train Multiple Models

In [ ]:
# ── Cell 13: Model training ───────────────────────────────────────────────────
models_config = {
    'RandomForest': RandomForestClassifier(
        n_estimators=150, max_depth=10, min_samples_leaf=4,
        class_weight='balanced', n_jobs=-1, random_state=SEED
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=5, random_state=SEED
    ),
    'ExtraTrees': ExtraTreesClassifier(
        n_estimators=150, max_depth=12, min_samples_leaf=4,
        class_weight='balanced', n_jobs=-1, random_state=SEED
    ),
}

trained_models = {}
for name, model in models_config.items():
    print(f'Training {name}…', end=' ', flush=True)
    model.fit(X_tr, y_tr)
    trained_models[name] = model
    print('✅')

## 8. Evaluate & Compare

In [ ]:
# ── Cell 14: Metrics ─────────────────────────────────────────────────────────
rows = []
for name, m in trained_models.items():
    yp = m.predict(X_te)
    rows.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_te, yp), 4),
        'Precision': round(precision_score(y_te, yp, average='weighted', zero_division=0), 4),
        'Recall':    round(recall_score(y_te, yp, average='weighted', zero_division=0), 4),
        'F1_Score':  round(f1_score(y_te, yp, average='weighted', zero_division=0), 4),
    })
    print(f'\n── {name} ──────────────────────────────────────────')
    print(classification_report(
        le_y.inverse_transform(y_te), le_y.inverse_transform(yp),
        target_names=LABEL_ORDER
    ))

metrics_df = pd.DataFrame(rows)
print('\n── Summary ─────────────────────────────────────────')
display(metrics_df)

In [ ]:
# ── Cell 15: Confusion matrices ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, m) in zip(axes, trained_models.items()):
    yp = m.predict(X_te)
    cm = confusion_matrix(y_te, yp)
    disp = ConfusionMatrixDisplay(cm, display_labels=LABEL_ORDER)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}', fontsize=11, fontweight='bold')
plt.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrices_all.png', dpi=150)
plt.show()

In [ ]:
# ── Cell 16: Model comparison chart ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics_df)); w = 0.2
for i, (col, c) in enumerate(zip(['Accuracy','Precision','Recall','F1_Score'],
                                   ['#2196F3','#FF5722','#4CAF50','#FF9800'])):
    ax.bar(x+i*w, metrics_df[col], w, label=col, color=c, edgecolor='white')
ax.set_xticks(x+1.5*w); ax.set_xticklabels(metrics_df['Model'])
ax.set_ylim(0, 1.12); ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/model_comparison.png', dpi=150)
plt.show()

## 9. Feature Importance

In [ ]:
# ── Cell 17: Feature importance ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = ['#2196F3','#FF5722','#4CAF50']
for ax, (name, m), c in zip(axes, trained_models.items(), colors):
    fi = pd.Series(m.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
    fi.plot(kind='barh', ax=ax, color=c)
    ax.set_title(f'{name}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Importance')
plt.suptitle('Feature Importances', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/feature_importances_all.png', dpi=150)
plt.show()

## 10. Select Best Model & Save

In [ ]:
# ── Cell 18: Best model selection ────────────────────────────────────────────
best_row   = metrics_df.loc[metrics_df['F1_Score'].idxmax()]
best_name  = best_row['Model']
best_model = trained_models[best_name]
print(f'🏆 Best Model: {best_name}')
print(f'   Accuracy : {best_row["Accuracy"]}')
print(f'   F1 Score : {best_row["F1_Score"]}')
print(f'   Precision: {best_row["Precision"]}')
print(f'   Recall   : {best_row["Recall"]}')

In [ ]:
# ── Cell 19: Save model.pkl ───────────────────────────────────────────────────
bundle = {
    'model':         best_model,
    'model_name':    best_name,
    'label_encoder': le_y,
    'cat_encoders':  encoders,
    'pci_scaler':    pci_scaler,
    'feature_cols':  FEATURE_COLS,
    'label_order':   LABEL_ORDER,
    'trained_at':    datetime.now().isoformat(),
}
PKL_PATH = f'{OUTPUT_DIR}/model.pkl'
with open(PKL_PATH, 'wb') as f:
    pickle.dump(bundle, f)
print(f'✅ model.pkl saved → {PKL_PATH}')

## 11. Prediction Pipeline

In [ ]:
# ── Cell 20: Prediction pipeline ────────────────────────────────────────────
import sys; sys.path.insert(0, '.')
# If running standalone, copy app_utils.py next to this notebook
from app_utils import load_model, predict_severity, recommend_action

bundle = load_model(PKL_PATH)

sample_input = {
    'location_type':              'Junction',
    'illegal_vehicle_count':      8,
    'traffic_volume':             900,
    'average_speed':              18.5,
    'parking_occupancy':          72.0,
    'road_width':                 12.0,
    'historical_violation_count': 120,
    'nearby_event':               1,
    'day_of_week':                5,        # Saturday
    'time_of_day':                'Evening',
}

result = predict_severity(sample_input, bundle)
print(json.dumps({k: v for k, v in result.items() if k != 'feature_vector'}, indent=2, default=str))

## 12. Recommendation Engine Demo

In [ ]:
# ── Cell 21: Recommendation engine ───────────────────────────────────────────
print('Enforcement Recommendations by Severity Level')
print('=' * 55)
for level in ['Low', 'Medium', 'High', 'Critical']:
    rec = recommend_action(level)
    print(f'{rec["icon"]}  {level:8s}  →  {rec["action"]}')
    print(f'           {rec["message"]}')
    flags = []
    if rec['notification']: flags.append('SMS/Notification')
    if rec['challan']:       flags.append('e-Challan')
    if rec['towing']:        flags.append('Towing')
    if rec['officer']:       flags.append('Officer Deployment')
    if flags: print(f'           Actions: {" | ".join(flags)}')
    print()